# LoRA Fine-Tuning (Training) — Albertina PT-PT (900M)

Fine-tunes `PORTULAN/albertina-900m-portuguese-ptpt-encoder` (`src.config.MODEL_REGISTRY["albertina_900m"]`) as a 10-class `Decisao`.

## Section 1 — Setup & Data Loading

Seeds every RNG (`src.runtime.set_all_seeds`) before touching the data or the model, per the project's reproducibility convention.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path("../..").resolve()))
from src.capture import capture_full_attention_subset, capture_logits_and_cls
from src.config import MODEL_REGISTRY
from src.data import build_datasets, load_label_map, load_splits
from src.evaluate import (
    classification_report_text,
    evaluate_predictions,
    full_test_report,
)
from src.finetune import history_to_dicts, train, training_args_payload
from src.finetune_lora import build_model_and_tokenizer
from src.plotting import plot_confusion_matrix, plot_per_class_f1_curves, plot_training_curves
from src.runtime import get_device, set_all_seeds

set_all_seeds()

In [ ]:
model_cfg = MODEL_REGISTRY["albertina_900m"]
device = get_device()

df = load_splits()
LABELS = load_label_map()["labels"]

print(f"device: {device}")
print(f"split sizes: {df['split'].value_counts().to_dict()}")
print(f"labels ({len(LABELS)}): {LABELS}")

## Section 2 — Step 1: Initialize the Model (LoRA + Gradient Checkpointing)

`src.finetune_lora.build_model_and_tokenizer` loads Albertina (900M) via `AutoModelForSequenceClassification` with `num_labels=10`.

In [ ]:
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

model, tokenizer = build_model_and_tokenizer(
    model_cfg, num_labels=len(LABELS), device=device,
    lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
)
model.print_trainable_parameters()

## Section 3 — Step 2: Configure the Training Engine

`src.data.build_datasets` tokenizes `Sumario` and attaches the numeric `label_id` for every row.

In [ ]:
datasets = build_datasets(df, tokenizer, max_length=model_cfg.max_length)
{split: len(ds) for split, ds in datasets.items()}

## Section 4 — Step 3: Set the Hyperparameters

The optimizer/schedule hyperparameters are sourced from `model_cfg` (`src.config.MODEL_REGISTRY["albertina_900m"]`).

In [ ]:
print(f"num_train_epochs: {model_cfg.num_train_epochs}")
print(f"learning_rate: {model_cfg.learning_rate}")
print(f"weight_decay: {model_cfg.weight_decay}")
print(f"warmup_ratio: {model_cfg.warmup_ratio}")
print(f"batch_size ({device.type}): {model_cfg.batch_size(device.type)}")
print(f"gradient_accumulation_steps: {model_cfg.gradient_accumulation_steps}")
print(f"early_stopping_patience: {model_cfg.early_stopping_patience}")
print(f"lora_r: {LORA_R}")
print(f"lora_alpha: {LORA_ALPHA}")
print(f"lora_dropout: {LORA_DROPOUT}")

## Section 5 — Step 4: Loss Calculation and Execution

`src.finetune.train` runs the AdamW + linear-schedule loop for up to 5 epochs.

In [ ]:
results_dir = model_cfg.results_dir("finetune_lora")
checkpoint_dir = results_dir / "checkpoint"

history = train(
    model,
    train_dataset=datasets["train"],
    val_dataset=datasets["val"],
    model_cfg=model_cfg,
    device=device,
    labels=LABELS,
    checkpoint_dir=checkpoint_dir,
)

In [ ]:
model.save_pretrained(checkpoint_dir)
tokenizer.save_pretrained(checkpoint_dir)

In [ ]:
batch_size = model_cfg.batch_size(device.type)

full_split_artifacts = capture_logits_and_cls(model, datasets["test"], device, batch_size=batch_size)
for name, array in full_split_artifacts.items():
    np.save(results_dir / f"{name}.npy", array)
    print(f"{name}.npy: {array.shape}")

In [ ]:
N_ATTENTION_SUBSET = 50

attention_artifacts = capture_full_attention_subset(
    model, datasets["test"], device, n_samples=N_ATTENTION_SUBSET
)
for name, array in attention_artifacts.items():
    np.save(results_dir / f"{name}.npy", array)
    print(f"{name}.npy: {array.shape}")

test_df = df[df["split"] == "test"].reset_index(drop=True)
attention_subset_ids = test_df["ImportJurisprudenciaId"].iloc[:N_ATTENTION_SUBSET].tolist()
with open(results_dir / "attention_subset_ids.json", "w", encoding="utf-8") as f:
    json.dump(attention_subset_ids, f, indent=2)

In [ ]:
test_label_ids = test_df["label_id"].tolist()
test_pred_ids = full_split_artifacts["logits"].argmax(axis=-1).tolist()
test_metrics = evaluate_predictions(test_label_ids, test_pred_ids, LABELS)
test_report = full_test_report(test_label_ids, test_pred_ids, LABELS)

metrics_payload = {
    "epoch_history": history_to_dicts(history),
    "test": test_metrics,
    "classification_report": test_report["classification_report"],
    "confusion_matrix": test_report["confusion_matrix"],
    "confusion_matrix_labels": test_report["confusion_matrix_labels"],
}
with open(results_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)

args_payload = training_args_payload(model_cfg, device, epochs_run=len(history))
args_payload["training_regime"] = "lora_finetune"
args_payload["gradient_checkpointing"] = True
args_payload["lora_r"] = LORA_R
args_payload["lora_alpha"] = LORA_ALPHA
args_payload["lora_dropout"] = LORA_DROPOUT
args_payload["lora_target_modules"] = "all-linear"
args_payload["lora_modules_to_save"] = ["classifier"]
with open(results_dir / "training_args.json", "w", encoding="utf-8") as f:
    json.dump(args_payload, f, indent=2)

with open(results_dir / "classification_report.txt", "w", encoding="utf-8") as f:
    f.write(classification_report_text(test_label_ids, test_pred_ids, LABELS))

plot_training_curves(history, results_dir / "training_curves.png", title=f"{model_cfg.display_name} — {results_dir.name}")

plot_per_class_f1_curves(history, LABELS, results_dir / "per_class_f1_curves.png", title=f"{model_cfg.display_name} — {results_dir.name}")

plot_confusion_matrix(
    test_report["confusion_matrix"], LABELS, results_dir / "confusion_matrix.png",
    title=f"{model_cfg.display_name} — {results_dir.name}",
)

test_metrics